In [1]:
from __future__ import annotations

# ============================================================
# PS S6E4 - UtaAzu LGBM experimental reproduction
# Revised after reviewing the attached notebook code.
# Goal:
# - stay close to the shared notebook skeleton
# - keep your experiment-friendly outputs
# - save raw / biased OOF and test probabilities
# ============================================================

In [2]:
import gc
import json
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path

import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
# ============================================================
# Config
# ============================================================
@dataclass
class CFG:
    COMP_DIR: str = "/kaggle/input/competitions/playground-series-s6e4/"
    ORIG_PATH: str = "/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv"

    TARGET: str = "Irrigation_Need"
    ID_COL: str = "id"
    RANDOM_SEED: int = 42
    N_FOLDS: int = 5
    N_BIAS_TRIALS: int = 100

    TAG: str = "utaazu_lgbm_min_repro"
    OUTDIR: str = "/kaggle/working/utaazu_lgbm_min_repro"


cfg = CFG()
OUTDIR = Path(cfg.OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)

TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

CATS = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Mulching_Used",
    "Region",
]
NUMS = [
    "Soil_pH",
    "Soil_Moisture",
    "Organic_Carbon",
    "Electrical_Conductivity",
    "Temperature_C",
    "Humidity",
    "Rainfall_mm",
    "Sunlight_Hours",
    "Wind_Speed_kmh",
    "Field_Area_hectare",
    "Previous_Irrigation_mm",
]

In [4]:
# ============================================================
# Utilities
# ============================================================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def save_json(path: Path, obj: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def public_preds(proba: np.ndarray, bias: np.ndarray) -> np.ndarray:
    return np.argmax(np.log(np.clip(proba, 1e-15, 1.0)) + bias, axis=1)


def get_calibrated_probs(probs: np.ndarray, bias: np.ndarray) -> np.ndarray:
    log_p = np.log(np.clip(probs, 1e-15, 1.0)) + bias
    exp_p = np.exp(log_p)
    return exp_p / exp_p.sum(axis=1, keepdims=True)


def get_rank_corr(a: np.ndarray, b: np.ndarray) -> float:
    ra = pd.Series(a).rank(method="average").to_numpy()
    rb = pd.Series(b).rank(method="average").to_numpy()
    return float(np.corrcoef(ra, rb)[0, 1])

In [5]:
# ============================================================
# Load data
# ============================================================
seed_everything(cfg.RANDOM_SEED)

train = pd.read_csv(Path(cfg.COMP_DIR) / "train.csv")
test = pd.read_csv(Path(cfg.COMP_DIR) / "test.csv")
orig = pd.read_csv(cfg.ORIG_PATH)

train["target_num"] = train[cfg.TARGET].map(TARGET_MAP)
orig["target_num"] = orig[cfg.TARGET].map(TARGET_MAP)

y_train = train["target_num"].values.astype(int)
test_id = test[cfg.ID_COL].copy()

print(f"Data Loaded. Train: {train.shape[0]:,}, Test: {test.shape[0]:,}, Original: {orig.shape[0]:,}")
print(f"Baseline High-Class Probability (Train): {(train['target_num'] == 2).mean():.2%}")

Data Loaded. Train: 630,000, Test: 270,000, Original: 10,000
Baseline High-Class Probability (Train): 3.33%


In [6]:
# ============================================================
# Feature engineering: close to attached notebook
# ============================================================
def apply_advanced_fe(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()

    # 1. Chris Deotte-style physical score
    h_score = (
        (d["Soil_Moisture"] < 25).astype(int) * 2
        + (d["Rainfall_mm"] < 300).astype(int) * 2
        + (d["Temperature_C"] > 30).astype(int)
        + (d["Wind_Speed_kmh"] > 10).astype(int)
    )
    l_score = (
        (d["Crop_Growth_Stage"] == "Harvest").astype(int) * 2
        + (d["Crop_Growth_Stage"] == "Sowing").astype(int) * 2
        + (d["Mulching_Used"] == "Yes").astype(int)
    )
    d["deotte_score"] = h_score - l_score

    # 2. Adaptive rounding from the shared notebook
    for col in NUMS:
        m = d[col].max()
        decimals = 3 if m < 10 else 2 if m < 100 else 1
        d[f"{col}_rounded"] = d[col].round(decimals)

    # 3. Forensic flags from the notebook EDA
    d["is_canyon_A"] = (d["Rainfall_mm"] < 463).astype(int)
    d["is_canyon_B"] = ((d["Rainfall_mm"] >= 1902) & (d["Rainfall_mm"] <= 2007)).astype(int)
    d["q3_rain_breach"] = ((d["Soil_Moisture"] < 25) & (d["Rainfall_mm"] >= 300)).astype(int)
    d["mulch_moisture_interaction"] = (d["Mulching_Used"] == "No").astype(int) * d["Soil_Moisture"]

    return d


print("Applying feature engineering...")
X = apply_advanced_fe(train.drop(columns=[cfg.ID_COL, cfg.TARGET, "target_num"]))
X_test = apply_advanced_fe(test.drop(columns=[cfg.ID_COL]))

te_cols = CATS + [c for c in X.columns if c.endswith("_rounded")]
base_cols = [c for c in X.columns if c not in te_cols]

pd.DataFrame({"feature": X.columns}).to_csv(OUTDIR / "feature_columns.csv", index=False)
print(f"Base feature count: {len(base_cols)} | TE source count: {len(te_cols)} | Total engineered count before TE concat: {X.shape[1]}")


Applying feature engineering...
Base feature count: 16 | TE source count: 19 | Total engineered count before TE concat: 35


In [7]:
# ============================================================
# Model params: match attached notebook much more closely
# ============================================================
lgb_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": 0.03,
    "num_leaves": 127,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "n_estimators": 2000,
    "random_state": cfg.RANDOM_SEED,
    "verbose": -1,
    "n_jobs": -1,
}

In [8]:
# ============================================================
# Training loop
# ============================================================
skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)
oof_probs = np.zeros((len(X), 3), dtype=np.float32)
test_probs = np.zeros((len(X_test), 3), dtype=np.float32)
fold_scores: list[float] = []
feature_importances: list[pd.DataFrame] = []

print("Training LightGBM with in-fold Target Encoding...")
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y_train), start=1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}/{cfg.N_FOLDS}")
    print("=" * 70)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y_train[tr_idx]
    X_va = X.iloc[val_idx].copy()
    y_va = y_train[val_idx]
    X_te = X_test.copy()

    encoder = TargetEncoder(target_type="multiclass", random_state=cfg.RANDOM_SEED)

    enc_tr = pd.DataFrame(
        encoder.fit_transform(X_tr[te_cols], y_tr).astype(np.float32),
        index=X_tr.index,
    )
    enc_va = pd.DataFrame(
        encoder.transform(X_va[te_cols]).astype(np.float32),
        index=X_va.index,
    )
    enc_te = pd.DataFrame(
        encoder.transform(X_te[te_cols]).astype(np.float32),
        index=X_te.index,
    )

    enc_col_names = [f"TE_{i}" for i in range(enc_tr.shape[1])]
    enc_tr.columns = enc_col_names
    enc_va.columns = enc_col_names
    enc_te.columns = enc_col_names

    X_tr_final = pd.concat([X_tr[base_cols].reset_index(drop=True), enc_tr.reset_index(drop=True)], axis=1)
    X_va_final = pd.concat([X_va[base_cols].reset_index(drop=True), enc_va.reset_index(drop=True)], axis=1)
    X_te_final = pd.concat([X_te[base_cols].reset_index(drop=True), enc_te.reset_index(drop=True)], axis=1)

    sw = compute_sample_weight("balanced", y_tr).astype(np.float32)

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr_final,
        y_tr,
        sample_weight=sw,
        eval_set=[(X_va_final, y_va)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

    va_pred = model.predict_proba(X_va_final)
    te_pred = model.predict_proba(X_te_final)
    oof_probs[val_idx] = va_pred.astype(np.float32)
    test_probs += te_pred.astype(np.float32) / cfg.N_FOLDS

    fold_ba = balanced_accuracy_score(y_va, np.argmax(va_pred, axis=1))
    fold_scores.append(float(fold_ba))
    print(f"Fold {fold} BA: {fold_ba:.6f}")

    fi = pd.DataFrame({
        "feature": X_tr_final.columns,
        f"gain_f{fold:02d}": model.booster_.feature_importance(importance_type="gain"),
        f"split_f{fold:02d}": model.booster_.feature_importance(importance_type="split"),
    })
    feature_importances.append(fi)

    np.save(OUTDIR / f"oof_{cfg.TAG}_partial_f{fold:02d}.npy", oof_probs)
    np.save(OUTDIR / f"pred_{cfg.TAG}_partial_f{fold:02d}.npy", te_pred.astype(np.float32))

    del X_tr, y_tr, X_va, y_va, X_te
    del enc_tr, enc_va, enc_te
    del X_tr_final, X_va_final, X_te_final
    del sw, model, va_pred, te_pred
    gc.collect()

raw_oof_ba = balanced_accuracy_score(y_train, np.argmax(oof_probs, axis=1))
print("\nRaw OOF Balanced Accuracy:", f"{raw_oof_ba:.6f}")
print("Mean fold BA:", f"{np.mean(fold_scores):.6f}")

Training LightGBM with in-fold Target Encoding...

Fold 1/5
Fold 1 BA: 0.974453

Fold 2/5
Fold 2 BA: 0.974871

Fold 3/5
Fold 3 BA: 0.976414

Fold 4/5
Fold 4 BA: 0.974945

Fold 5/5
Fold 5 BA: 0.974225

Raw OOF Balanced Accuracy: 0.974982
Mean fold BA: 0.974982


In [9]:
# ============================================================
# Bias tuning: revised to match attached notebook (Optuna)
# ============================================================
print("\nBias Tuning via Optuna...")

def objective(trial: optuna.Trial) -> float:
    b = np.array([trial.suggest_float(f"b{i}", -2.0, 2.0) for i in range(3)], dtype=np.float64)
    return balanced_accuracy_score(y_train, public_preds(oof_probs, b))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=cfg.N_BIAS_TRIALS, show_progress_bar=False)

best_bias = np.array([study.best_params[f"b{i}"] for i in range(3)], dtype=np.float64)
tuned_ba = float(study.best_value)

print(f"Raw OOF Balanced Accuracy   : {raw_oof_ba:.6f}")
print(f"Tuned OOF Balanced Accuracy : {tuned_ba:.6f}")
print("Best bias:", best_bias)


Bias Tuning via Optuna...
Raw OOF Balanced Accuracy   : 0.974982
Tuned OOF Balanced Accuracy : 0.979323
Best bias: [-0.6856312  -0.72419563  1.5955005 ]


In [10]:
# ============================================================
# Final calibrated probabilities and saves
# ============================================================
oof_probs_final = get_calibrated_probs(oof_probs, best_bias).astype(np.float32)
test_probs_final = get_calibrated_probs(test_probs, best_bias).astype(np.float32)

# raw save
np.save(OUTDIR / f"oof_{cfg.TAG}_proba.npy", oof_probs)
np.save(OUTDIR / f"pred_{cfg.TAG}_proba.npy", test_probs)

# biased save
np.save(OUTDIR / f"oof_{cfg.TAG}_proba_biased.npy", oof_probs_final)
np.save(OUTDIR / f"pred_{cfg.TAG}_proba_biased.npy", test_probs_final)
np.save(OUTDIR / "best_class_bias.npy", best_bias)

raw_preds = np.argmax(test_probs, axis=1)
final_preds = public_preds(test_probs, best_bias)

sub_raw = pd.DataFrame({
    cfg.ID_COL: test_id,
    cfg.TARGET: [INV_TARGET_MAP[p] for p in raw_preds],
})
sub_bias = pd.DataFrame({
    cfg.ID_COL: test_id,
    cfg.TARGET: [INV_TARGET_MAP[p] for p in final_preds],
})
sub_raw.to_csv(OUTDIR / f"submission_{cfg.TAG}.csv", index=False)
sub_bias.to_csv(OUTDIR / "submission.csv", index=False)
sub_bias.to_csv(OUTDIR / f"submission_{cfg.TAG}_biased.csv", index=False)

# feature importance merge
fi_merged = feature_importances[0]
for part in feature_importances[1:]:
    fi_merged = fi_merged.merge(part, on="feature", how="outer")

gain_cols = [c for c in fi_merged.columns if c.startswith("gain_f")]
split_cols = [c for c in fi_merged.columns if c.startswith("split_f")]
fi_merged["importance_gain_mean"] = fi_merged[gain_cols].fillna(0).mean(axis=1)
fi_merged["importance_split_mean"] = fi_merged[split_cols].fillna(0).mean(axis=1)
fi_merged = fi_merged.sort_values("importance_gain_mean", ascending=False)
fi_merged.to_csv(OUTDIR / "feature_importance.csv", index=False)

cv_result = {
    "tag": cfg.TAG,
    "raw_cv": float(raw_oof_ba),
    "biased_cv": float(tuned_ba),
    "fold_scores": [float(x) for x in fold_scores],
    "best_bias": [float(x) for x in best_bias],
    "n_folds": cfg.N_FOLDS,
    "n_bias_trials": cfg.N_BIAS_TRIALS,
    "te_cols": te_cols,
    "base_cols": base_cols,
    "lgb_params": lgb_params,
}
save_json(OUTDIR / "cv_result.json", cv_result)

print("\nSaved files:")
print(OUTDIR / f"oof_{cfg.TAG}_proba.npy")
print(OUTDIR / f"pred_{cfg.TAG}_proba.npy")
print(OUTDIR / f"oof_{cfg.TAG}_proba_biased.npy")
print(OUTDIR / f"pred_{cfg.TAG}_proba_biased.npy")
print(OUTDIR / "best_class_bias.npy")
print(OUTDIR / "feature_columns.csv")
print(OUTDIR / "feature_importance.csv")
print(OUTDIR / "cv_result.json")
print(OUTDIR / f"submission_{cfg.TAG}.csv")
print(OUTDIR / f"submission_{cfg.TAG}_biased.csv")
print(OUTDIR / "submission.csv")


Saved files:
/kaggle/working/utaazu_lgbm_min_repro/oof_utaazu_lgbm_min_repro_proba.npy
/kaggle/working/utaazu_lgbm_min_repro/pred_utaazu_lgbm_min_repro_proba.npy
/kaggle/working/utaazu_lgbm_min_repro/oof_utaazu_lgbm_min_repro_proba_biased.npy
/kaggle/working/utaazu_lgbm_min_repro/pred_utaazu_lgbm_min_repro_proba_biased.npy
/kaggle/working/utaazu_lgbm_min_repro/best_class_bias.npy
/kaggle/working/utaazu_lgbm_min_repro/feature_columns.csv
/kaggle/working/utaazu_lgbm_min_repro/feature_importance.csv
/kaggle/working/utaazu_lgbm_min_repro/cv_result.json
/kaggle/working/utaazu_lgbm_min_repro/submission_utaazu_lgbm_min_repro.csv
/kaggle/working/utaazu_lgbm_min_repro/submission_utaazu_lgbm_min_repro_biased.csv
/kaggle/working/utaazu_lgbm_min_repro/submission.csv


In [11]:
# ============================================================
# Optional helper for correlation checks against existing OOFs
# ============================================================
def compare_with_existing_candidate(path_a: str, path_b: str) -> dict:
    a = np.load(path_a)
    b = np.load(path_b)
    if a.shape != b.shape:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")
    out = {}
    for i in range(a.shape[1]):
        out[f"class_{i}_raw_corr"] = float(np.corrcoef(a[:, i], b[:, i])[0, 1])
        out[f"class_{i}_rank_corr"] = get_rank_corr(a[:, i], b[:, i])
    return out

print("\nDone.")


Done.
